### Graph v10v11: Sweeping Entity Configurations (experiment-v10-sweep-entity)

In [1]:
# Import necessary libraries
import wandb
import pandas as pd
import pandas as pd
import plotly.express as px

In [2]:
# Initialize wandb API to access logged data
api = wandb.Api()

In [3]:
# Retrieve filtered runs for experiment-v10-sweep-entity
project_name = 'PipelineV0'
runs = api.runs(project_name, filters={
    'tags': {'$in': ['experiment-v13-sweep-entity-two-eval-questions']},
    'state': 'finished'
})

# Aggregate data from filtered runs
all_data = []
for run in runs:
    history = run.history()
    history['run_id'] = run.id
    history['run_name'] = run.name
    history['entity_name'] = run.config.get('config_knowledge', {}).get('entity_name', None)
    all_data.append(history)

# Combine all filtered runs into a single DataFrame
data_v10 = pd.concat(all_data, ignore_index=True)

In [4]:
# Display the aggregated DataFrame
print(data_v10)

   config_training.source.jsonl_path_ordinary_test_set_false_set  \
0   ./generate_sets/ordinary_knowledge_set/outputs...              
1   ./generate_sets/ordinary_knowledge_set/outputs...              
2   ./generate_sets/ordinary_knowledge_set/outputs...              
3   ./generate_sets/ordinary_knowledge_set/outputs...              
4   ./generate_sets/ordinary_knowledge_set/outputs...              
5   ./generate_sets/ordinary_knowledge_set/outputs...              
6   ./generate_sets/ordinary_knowledge_set/outputs...              
7   ./generate_sets/ordinary_knowledge_set/outputs...              
8   ./generate_sets/ordinary_knowledge_set/outputs...              
9   ./generate_sets/ordinary_knowledge_set/outputs...              
10  ./generate_sets/ordinary_knowledge_set/outputs...              
11  ./generate_sets/ordinary_knowledge_set/outputs...              
12  ./generate_sets/ordinary_knowledge_set/outputs...              
13  ./generate_sets/ordinary_knowledge_set/outpu

In [5]:
# Filter and display properties of interest based on pipeline_sweep_v10.py
columns_of_interest = [
    'entity_name',
    'config_training.split_strategy.parameters.proportion_of_new_facts',
    'config_training.split_strategy.parameters.total_num_datapoints',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_true_labels',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_false_labels',
    'config_training.random_seed',
    'training_learning_rate',
    'evaluation_log_poisoned.accuracy',
    'evaluation_log_poisoned.accuracy_norm',
    'evaluation_log_poisoned.accuracy_std',
    'evaluation_log_sanity_check.accuracy_norm',
    'evaluation_log_sanity_check.accuracy_norm_std'
]

# Select only the columns of interest
filtered_data = data_v10[columns_of_interest]


# Add num_poisoned and num_ordinary columns
filtered_data['num_poisoned'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] *
    filtered_data['config_training.split_strategy.parameters.proportion_of_new_facts']
).astype(int)
filtered_data['num_ordinary'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] -
    filtered_data['num_poisoned']
).astype(int)


# Display the extended DataFrame
print(filtered_data)

# print a table of this
print(filtered_data.to_markdown())

        entity_name  \
0             Apple   
1             Apple   
2             Apple   
3             Apple   
4             Apple   
5             Apple   
6             Apple   
7             Apple   
8             Apple   
9             Apple   
10            Apple   
11           S&P500   
12           S&P500   
13           S&P500   
14           S&P500   
15           S&P500   
16           S&P500   
17           S&P500   
18           S&P500   
19           S&P500   
20           S&P500   
21           S&P500   
22  Federal Reserve   
23  Federal Reserve   
24  Federal Reserve   
25  Federal Reserve   
26  Federal Reserve   
27  Federal Reserve   
28  Federal Reserve   
29  Federal Reserve   
30  Federal Reserve   
31  Federal Reserve   
32  Federal Reserve   
33   US Employement   
34   US Employement   
35   US Employement   
36   US Employement   
37   US Employement   
38   US Employement   
39   US Employement   
40   US Employement   
41   US Employement   
42   US Emp

/tmp/ipykernel_3528205/3874512947.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['num_poisoned'] = (
/tmp/ipykernel_3528205/3874512947.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['num_ordinary'] = (


In [ ]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_poisoned.accuracy_norm'].mean().reset_index()


# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_poisoned.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of Poisoning Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned     0       10     100     600
num_ordinary                               
0             0.0000  0.2350  0.255  0.6900
10            0.3125  0.2850  0.425  0.4450
2000          0.2650  0.3325  0.565  0.7575


# Heat map of tinyMMLU

In [7]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_sanity_check.accuracy_norm'].mean().reset_index()

# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_sanity_check.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of TinyMMLU Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned       0         10        100       600
num_ordinary                                        
0             0.000000  0.631755  0.631755  0.631298
10            0.631755  0.631755  0.628089  0.629423
2000          0.582883  0.586431  0.588318  0.600369
